In [ ]:
import csv
import os
import re
from tqdm import tqdm
import logging
import pickle
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator


import seaborn as sns
import warnings
from scipy.stats import pearsonr

import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import ukbiobank
import ukbiobank.utils.utils
from ukbiobank.utils import loadCsv
from ukbiobank.utils import addFields
from ukbiobank.utils.utils import fieldIdsToNames

# 1. Accelerometry

In [ ]:
# Upload UK Bioabank csv
csv_path = '/UK_BB/ukbbdata/ukbb_oct23/ukb675684.csv'
ukb = ukbiobank.ukbio(ukb_csv=csv_path)

In [ ]:
df_accelerometry = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
40048, #Light - Overall average
40049, #Moderate-Vigorous - Overall average
40047, #Sedentary - Overall average
40046, #Sleep - Overall average
#Acceleration averages
90012, #Overall acceleration average
90013, #Standard deviation of acceleration
90027, #Average acceleration 00:00 - 00:59
90028, #Average acceleration 01:00 - 01:59
90029, #Average acceleration 02:00 - 02:59
90030, #Average acceleration 03:00 - 03:59
90031, #Average acceleration 04:00 - 04:59
90032, #Average acceleration 05:00 - 05:59
90033, #Average acceleration 06:00 - 06:59
90034, #Average acceleration 07:00 - 07:59
90035, #Average acceleration 08:00 - 08:59
90036, #Average acceleration 09:00 - 09:59
90037, #Average acceleration 10:00 - 10:59
90038, #Average acceleration 11:00 - 11:59
90039, #Average acceleration 12:00 - 12:59
90040, #Average acceleration 13:00 - 13:59
90041, #Average acceleration 14:00 - 14:59
90042, #Average acceleration 15:00 - 15:59
90043, #Average acceleration 16:00 - 16:59
90044, #Average acceleration 17:00 - 17:59
90045, #Average acceleration 18:00 - 18:59
90046, #Average acceleration 19:00 - 19:59
90047, #Average acceleration 20:00 - 20:59
90048, #Average acceleration 21:00 - 21:59
90049, #Average acceleration 22:00 - 22:59
90050, #Average acceleration 23:00 - 23:59
90019, #Monday average acceleration
90020, #Tuesday average acceleration
90021, #Wednesday average acceleration
90022, #Thursday average acceleration
90023, #Friday average acceleration
90024, #Saturday average acceleration
90025, #Sunday average acceleration
90087, #No-wear time bias adjusted average acceleration
90088, #No-wear time bias adjusted acceleration standard deviation
90091, #No-wear time bias adjusted acceleration maximum
90090, #No-wear time bias adjusted acceleration minimum
90089, #No-wear time bias adjusted acceleration median
#Acceleration intensity distribution
90092, #Fraction acceleration <= 1 milli-gravities
90093, #Fraction acceleration <= 2 milli-gravities
90094, #Fraction acceleration <= 3 milli-gravities
90095, #Fraction acceleration <= 4 milli-gravities
90096, #Fraction acceleration <= 5 milli-gravities
90097, #Fraction acceleration <= 6 milli-gravities
90098, #Fraction acceleration <= 7 milli-gravities
90099, #Fraction acceleration <= 8 milli-gravities
90100, #Fraction acceleration <= 9 milli-gravities
90101, #Fraction acceleration <= 10 milli-gravities
90102, #Fraction acceleration <= 11 milli-gravities
90103, #Fraction acceleration <= 12 milli-gravities
90104, #Fraction acceleration <= 13 milli-gravities
90105, #Fraction acceleration <= 14 milli-gravities
90106, #Fraction acceleration <= 15 milli-gravities
90107, #Fraction acceleration <= 16 milli-gravities
90108, #Fraction acceleration <= 17 milli-gravities
90109, #Fraction acceleration <= 18 milli-gravities
90110, #Fraction acceleration <= 19 milli-gravities
90111, #Fraction acceleration <= 20 milli-gravities
90112, #Fraction acceleration <= 25 milli-gravities
90113, #Fraction acceleration <= 30 milli-gravities
90114, #Fraction acceleration <= 35 milli-gravities
90115, #Fraction acceleration <= 40 milli-gravities
90116, #Fraction acceleration <= 45 milli-gravities
90117, #Fraction acceleration <= 50 milli-gravities
90118, #Fraction acceleration <= 55 milli-gravities
90119, #Fraction acceleration <= 60 milli-gravities
90120, #Fraction acceleration <= 65 milli-gravities
90121, #Fraction acceleration <= 70 milli-gravities
90122, #Fraction acceleration <= 75 milli-gravities
90123, #Fraction acceleration <= 80 milli-gravities
90124, #Fraction acceleration <= 85 milli-gravities
90125, #Fraction acceleration <= 90 milli-gravities
90126, #Fraction acceleration <= 95 milli-gravities
90127, #Fraction acceleration <= 100 milli-gravities
90128, #Fraction acceleration <= 125 milli-gravities
90129, #Fraction acceleration <= 150 milli-gravities
90130, #Fraction acceleration <= 175 milli-gravities
90131, #Fraction acceleration <= 200 milli-gravities
90132, #Fraction acceleration <= 225 milli-gravities
90133, #Fraction acceleration <= 250 milli-gravities
90134, #Fraction acceleration <= 275 milli-gravities
90135, #Fraction acceleration <= 300 milli-gravities
90136, #Fraction acceleration <= 325 milli-gravities
90137, #Fraction acceleration <= 350 milli-gravities
90138, #Fraction acceleration <= 375 milli-gravities
90139, #Fraction acceleration <= 400 milli-gravities
90140, #Fraction acceleration <= 425 milli-gravities
90141, #Fraction acceleration <= 450 milli-gravities
90142, #Fraction acceleration <= 475 milli-gravities
90143, #Fraction acceleration <= 500 milli-gravities
90144, #Fraction acceleration <= 600 milli-gravities
90145, #Fraction acceleration <= 700 milli-gravities
90146, #Fraction acceleration <= 800 milli-gravities
90147, #Fraction acceleration <= 900 milli-gravities
90148, #Fraction acceleration <= 1000 milli-gravities
90149, #Fraction acceleration <= 1100 milli-gravities
90150, #Fraction acceleration <= 1200 milli-gravities
90151, #Fraction acceleration <= 1300 milli-gravities
90152, #Fraction acceleration <= 1400 milli-gravities
90153, #Fraction acceleration <= 1500 milli-gravities
90154, #Fraction acceleration <= 1600 milli-gravities
90155, #Fraction acceleration <= 1700 milli-gravities
90156, #Fraction acceleration <= 1800 milli-gravities
90157, #Fraction acceleration <= 1900 milli-gravities
90158, #Fraction acceleration <= 2000 milli-gravities
#Accelerometer wear time duration
90057, #Wear duration during Friday
90053, #Wear duration during Monday
90058, #Wear duration during Saturday
90059, #Wear duration during Sunday
90056, #Wear duration during Thursday
90054, #Wear duration during Tuesday
90055, #Wear duration during Wednesday
90051, #Wear duration overall
#Accelerometer calibration
90016, #Data quality, good calibration
90017, #Data quality, calibrated on own data
90159, #Error tolerance before calibration
90160, #Error tolerance after calibration
90170, #Calibration coefficients - mean temperature
90171, #Calibration - number of static points
90173, #Calibration - maximum x stationary value
90175, #Calibration - maximum y stationary value
90177, #Calibration - maximum z stationary value
90172, #Calibration - minimum x stationary value
90174, #Calibration - minimum y stationary value
90176, #Calibration - minimum z stationary value
90161, #Calibration coefficients - x offset
90164, #Calibration coefficients - x slope
90167, #Calibration coefficients - x temp
90162, #Calibration coefficients - y offset
90165, #Calibration coefficients - y slope
90168, #Calibration coefficients - y temp
90163, #Calibration coefficients - z offset
90166, #Calibration coefficients - z slope
90169, #Calibration coefficients - z temp
#Raw accelerometer statistics
90002, #Data problem indicator
90182, #Data recording errors
90179, #Device ID
90187, #Total data readings
90188, #Sample rate average
90189, #Sample rate standard deviation
90190, #Sample rate minimum
90191, #Sample rate maximum
90180, #Interrupted recording periods
90181, #Duration of interrupted recording periods
90183, #Readings exceeding +/-8 gravities before calibration
90185, #Readings exceeding +/-8 gravities after calibration
90184, #Maximum readings exceeding +/-8 gravities before calibration in a 5 second epoch
90186, #Maximum readings exceeding +/-8 gravities after calibration in a 5 second epoch
90192, #Temperature average
90193, #Temperature standard deviation
90194, #Temperature minimum
90195, #Temperature maximum
])
df_accelerometry_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_accelerometry)
df_accelerometry_names.to_csv('/UK_BB/brainbody/activity/data/accelerometry_vars.csv', index=False)

# Get remaining variables
activity_pattern = pd.read_csv('/UK_BB/ukbbdata/ukbb_oct23/ukb675684.csv', usecols=lambda col: col == 'eid' or col.startswith('40046-') or col.startswith('40049-') or col.startswith('40047-') or col.startswith('40048-') or col.startswith('40049-'))
activity_pattern.to_csv('/UK_BB/brainbody/activity/data/activity_pattern.csv', index=False)

In [ ]:
# 40046, #Sleep - Overall average
# 40047, #Sedentary - Overall average
# 40048, #Light - Overall average
# 40049, #Moderate-Vigorous - Overall average

In [ ]:
# Combine features and save
df_accelerometry = pd.read_csv('/UK_BB/brainbody/activity/data/accelerometry_vars.csv')
activity_pattern.columns = activity_pattern.columns.str.replace('40046-0.0', 'Sleep - Overall average-0.0')
activity_pattern.columns = activity_pattern.columns.str.replace('40047-0.0', 'Sedentary - Overall average-0.0')
activity_pattern.columns = activity_pattern.columns.str.replace('40048-0.0', 'Light - Overall average-0.0')
activity_pattern.columns = activity_pattern.columns.str.replace('40049-0.0', 'Moderate-Vigorous - Overall average-0.0')

# Merge data
accelerometry = df_accelerometry.merge(activity_pattern, on = 'eid')
# Save
accelerometry.to_csv('/UK_BB/brainbody/activity/data/accelerometry_raw.csv', index=False)

In [ ]:
# Filter Instance 0
accelerometry = pd.read_csv('/UK_BB/brainbody/activity/data/accelerometry_raw.csv')
accelerometry_i0 = accelerometry[['eid'] + accelerometry.filter(regex=r'0\.\d$').columns.tolist()]
accelerometry_i0.columns = accelerometry_i0.columns.str.replace('-0.0', '')
accelerometry_i0.columns.to_list()

In [ ]:
accelerometry_all_vars_i0 = accelerometry_i0.drop(columns=[
 'Data problem indicator',
 'Data quality, good calibration',
 'Data quality, calibrated on own data',
 'Error tolerance before calibration',
 'Error tolerance after calibration',
 'Calibration coefficients - x offset',
 'Calibration coefficients - y offset',
 'Calibration coefficients - z offset',
 'Calibration coefficients - x slope',
 'Calibration coefficients - y slope',
 'Calibration coefficients - z slope',
 'Calibration coefficients - x temp',
 'Calibration coefficients - y temp',
 'Calibration coefficients - z temp',
 'Calibration coefficients - mean temperature',
 'Calibration - number of static points',
 'Calibration - minimum x stationary value',
 'Calibration - maximum x stationary value',
 'Calibration - minimum y stationary value',
 'Calibration - maximum y stationary value',
 'Calibration - minimum z stationary value',
 'Calibration - maximum z stationary value',
 'Device ID',
 'Interrupted recording periods',
 'Duration of interrupted recording periods',
 'Data recording errors',
 'Readings exceeding +/-8 gravities before calibration',
 'Maximum readings exceeding +/-8 gravities before calibration in a 5 second epoch',
 'Readings exceeding +/-8 gravities after calibration',
 'Maximum readings exceeding +/-8 gravities after calibration in a 5 second epoch',
 'Total data readings',
 'Sample rate average',
 'Sample rate standard deviation',
 'Sample rate minimum',
 'Sample rate maximum',
 'Temperature average',
 'Temperature standard deviation',
 'Temperature minimum',
 'Temperature maximum']).dropna(axis=0).reset_index(drop=True)

print(accelerometry_all_vars_i0.shape)

accelerometry_all_vars_i0.to_csv('/UK_BB/brainbody/activity/data/accelerometry_all_vars_i0_nona.csv', index=False)
accelerometry_all_vars_i0.to_csv('/UK_BB/brainbody/activity/data/accelerometry_full.csv', index=False)
accelerometry_all_vars_i0.to_csv('/UK_BB/brainbody/activity/data/accelerometry.csv', index=False)

#Display max, min, and negative values
print('Shape', accelerometry_all_vars_i0.shape)
print('MIN\n', accelerometry_all_vars_i0.min().round(3))
print('MAX\n', accelerometry_all_vars_i0.max())
print('NEG\n', (accelerometry_all_vars_i0 < 0).sum().sort_values(ascending=False))
print('NA\n', (accelerometry_all_vars_i0.isna()).sum().sort_values(ascending=False))

# Self-reported activity

In [ ]:
df_activity_behav = ukbiobank.utils.utils.loadCsv(ukbio=ukb, fields=['eid',
#Physical activity (Touchscreen ⏵ Lifestyle and environment)
1100, #Drive faster than motorway speed limit
2634, #Duration of heavy DIY
1021, #Duration of light DIY
894, #Duration of moderate activity
3647, #Duration of other exercises
1001, #Duration of strenuous sports
914, #Duration of vigorous activity
874, #Duration of walks
981, #Duration walking for pleasure
2624, #Frequency of heavy DIY in last 4 weeks
1011, #Frequency of light DIY in last 4 weeks
3637, #Frequency of other exercises in last 4 weeks
943, #Frequency of stair climbing in last 4 weeks
991, #Frequency of strenuous sports in last 4 weeks
971, #Frequency of walking for pleasure in last 4 weeks
884, #Number of days/week of moderate physical activity 10+ minutes
904, #Number of days/week of vigorous physical activity 10+ minutes
864, #Number of days/week walked 10+ minutes
1090, #Time spent driving
1080, #Time spent using computer
1070, #Time spent watching television (TV)
6164, #Types of physical activity in last 4 weeks
6162, #Types of transport used (excluding work)
924, #Usual walking pace
#MET (Metabolic Equivalent Task) scores data based on IPAQ (International Physical Activity Questionnaire) guidelines
22035, #At or above moderate/vigorous recommendation
22036, #At or above moderate/vigorous/walking recommendation
22032, #IPAQ activity group
22038, #MET minutes per week for moderate activity
22039, #MET minutes per week for vigorous activity
22037, #MET minutes per week for walking
22040, #Summed MET minutes per week for all activity
22033, #Summed days activity
22034, #Summed minutes activity
#Online follow-up ⏵ Diet by 24-hour recall ⏵ Physical activity yesterday
104900, #Time spent doing vigorous physical activity
104910, #Time spent doing moderate physical activity
104920 #Time spent doing light physical activity
])
df_activity_behav_names = ukbiobank.utils.utils.fieldIdsToNames(ukbio=ukb, df=df_activity_behav)
df_activity_behav_names.to_csv('/UK_BB/brainbody/activity/data/activity_behav_vars.csv', index=False)

In [ ]:
print('% missing')
with pd.option_context('display.max_rows', None):
    display(((df_activity_behav_names.isna().sum() / len(df_activity_behav_names)).round(2)*100).sort_values(ascending=False))

Because each non-numeric response will be dummy-encoded, for questions with more than 1 arrays (ie types of transport use - allows more than 1 response) only use the first answer!

In [ ]:
# Filter Instances 0 and 2
activity_behav_i0_i2 = df_activity_behav_names[['eid'] + df_activity_behav_names.filter(regex=r'0\.\d$|2\.\d$').columns.tolist()]
activity_behav_i0_i2.to_csv('/UK_BB/brainbody/activity/data/activity_behav_i0_i2.csv', index=False)
with pd.option_context('display.max_rows', None):
    display(((activity_behav_i0_i2.isna().sum() / len(activity_behav_i0_i2)).round(2)*100).sort_values(ascending=False))

In [ ]:
# Filter Instances 3 and 4
activity_behav_i3_i4 = df_activity_behav_names[['eid'] + df_activity_behav_names.filter(regex=r'3\.\d$|4\.\d$').columns.tolist()]
activity_behav_i3_i4.to_csv('/UK_BB/brainbody/activity/data/activity_behav_i3_i4.csv', index=False)
with pd.option_context('display.max_rows', None):
    display(((activity_behav_i3_i4.isna().sum() / len(activity_behav_i3_i4)).round(2)*100).sort_values(ascending=False))

Go through each variable and select an appropriate Instance

In [ ]:
# Select Instance 2
activity_behav_i0_i2 = pd.read_csv('/UK_BB/brainbody/activity/data/activity_behav_i0_i2.csv')
activity_behav_i2 = activity_behav_i0_i2.drop(columns=[
#Physical activity (Touchscreen) Instance 0
'Usual walking pace-0.0',
'Types of transport used (excluding work)-0.0',
'Types of transport used (excluding work)-0.1',
'Types of transport used (excluding work)-0.2',
'Types of transport used (excluding work)-0.3',
'Types of physical activity in last 4 weeks-0.0',
'Types of physical activity in last 4 weeks-0.1',
'Types of physical activity in last 4 weeks-0.2',
'Types of physical activity in last 4 weeks-0.3',
'Types of physical activity in last 4 weeks-0.4',
'Time spent watching television (TV)-0.0',
'Time spent using computer-0.0',
'Time spent driving-0.0',
'Number of days/week walked 10+ minutes-0.0',
'Number of days/week of vigorous physical activity 10+ minutes-0.0',
'Number of days/week of moderate physical activity 10+ minutes-0.0',
'Frequency of walking for pleasure in last 4 weeks-0.0',
'Frequency of strenuous sports in last 4 weeks-0.0',
'Frequency of stair climbing in last 4 weeks-0.0',
'Frequency of other exercises in last 4 weeks-0.0',
'Frequency of light DIY in last 4 weeks-0.0',
'Frequency of heavy DIY in last 4 weeks-0.0',
'Duration walking for pleasure-0.0',
'Duration of walks-0.0',
'Duration of vigorous activity-0.0',
'Duration of strenuous sports-0.0',
'Duration of other exercises-0.0',
'Duration of moderate activity-0.0',
'Duration of light DIY-0.0',
'Duration of heavy DIY-0.0',
'Drive faster than motorway speed limit-0.0',
#MET (Metabolic Equivalent Task) scores Instance 0
'Summed minutes activity-0.0',
'Summed days activity-0.0',
'Summed MET minutes per week for all activity-0.0',
'MET minutes per week for walking-0.0',
'MET minutes per week for vigorous activity-0.0',
'MET minutes per week for moderate activity-0.0',
'IPAQ activity group-0.0',
'Above moderate/vigorous/walking recommendation-0.0',
'Above moderate/vigorous recommendation-0.0',
#Physical activity yesterday (online follow-up)
'Time spent doing vigorous physical activity-0.0',
'Time spent doing moderate physical activity-0.0',
'Time spent doing light physical activity-0.0'])

# 2. MET (Metabolic Equivalent Task) scores

In [ ]:
# MET (Metabolic Equivalent Task) scores
activity_met = activity_behav_i2[['eid',
'Summed minutes activity-2.0',
'Summed days activity-2.0',
'Summed MET minutes per week for all activity-2.0',
'MET minutes per week for walking-2.0',
'MET minutes per week for vigorous activity-2.0',
'MET minutes per week for moderate activity-2.0',
'IPAQ activity group-2.0',
'Above moderate/vigorous/walking recommendation-2.0',
'Above moderate/vigorous recommendation-2.0'
]]

# Count misssing values
print('% missing', ((activity_met.isna().sum() / len(activity_met)).round(2)*100).sort_values(ascending=False))
print('Sample size after removing NAs', activity_met.dropna(axis=0).shape)


activity_met = activity_met.dropna(axis=0).reset_index(drop=True)
activity_met.columns = activity_met.columns.str.replace('-2.0', '')

# Count negative values, min, and max
print('MIN\n', activity_met.min())
print('MAX\n', activity_met.max())
print('NEG\n', (activity_met < 0).sum().sort_values(ascending=False))

activity_met.to_csv('/UK_BB/brainbody/activity/data/activity_behav_MET_vars_i2_nona.csv', index=False)
activity_met.to_csv('/UK_BB/brainbody/activity/data/activity_MET.csv', index=False)

# 3. Yesterday activity, online follow-up: Physical activity by recall

#### 2.1. Online follow-up: Physical activity yesterday - Instance 2

In [ ]:
#Online follow-up: Physical activity yesterday - Instance 2
activity_yesterday = activity_behav_i2[['eid',
'Time spent doing vigorous physical activity-2.0',
'Time spent doing moderate physical activity-2.0',
'Time spent doing light physical activity-2.0'
]]

# Count misssing values
print('% missing', ((activity_yesterday.isna().sum() / len(activity_yesterday)).round(2)*100).sort_values(ascending=False))
print('Sample size after removing NAs', activity_yesterday.dropna(axis=0).shape)

# Count negative values, min, and max
print('MIN\n', activity_yesterday.min())
print('MAX\n', activity_yesterday.max())
print('NEG\n', (activity_yesterday < 0).sum().sort_values(ascending=False))

Change encoding for 'Physical activity yesterday'

1. Time spent doing vigorous physical activity / Time spent doing moderate physical activity

- 0 None -> 0
- 10 Under 10 minutes -> 1
- 12 1-2 hours -> 2
- 24 2-4 hours -> 3
- 46 4-6 hours -> 4
- 600 6+ hours -> 5
- 1030 10-30 minutes -> 6
- 3060 30-60 minutes -> 7

2. Time spent doing light physical activity

- 0 None -> 0
- 1 Under 1 hour -> 1
- 13 1-3 hours -> 2
- 35 3-5 hours -> 3
- 57 5-7 hours -> 4
- 79 7-9 hours -> 5
- 912 9-12 hours -> 6
- 1200 12+ hours -> 7

In [ ]:
# Change response encoding
activity_yesterday.loc[:, [
    'Time spent doing vigorous physical activity-2.0',
    'Time spent doing moderate physical activity-2.0']] = activity_yesterday.loc[:, [
    'Time spent doing vigorous physical activity-2.0',
    'Time spent doing moderate physical activity-2.0']].replace({
        #0: None - ok
        10: 1, #Under 10 minutes
        1030: 2, #10-30 minutes
        3060: 3, #30-60 minutes
        12: 4, #1-2 hours
        24: 5, #2-4 hours
        46: 6, #4-6 hours
        600: 7 #6+ hours

})

activity_yesterday.loc[:, [
    'Time spent doing light physical activity-2.0']] = activity_yesterday.loc[:, [
    'Time spent doing light physical activity-2.0']].replace({
        #0: None - ok
        #1: Under 1 hour - ok
        13: 2, #1-3 hours
        35: 3, #3-5 hours
        57: 4, #5-7 hours
        79: 5,  #7-9 hours
        912: 6, #9-12 hours
        1200: 7 #12+ hours
})

activity_yesterday = activity_yesterday.dropna(axis=0)
activity_yesterday.columns = activity_yesterday.columns.str.replace('-2.0', '')
# Count negative values, min, and max
print('MIN\n', activity_yesterday.min())
print('MAX\n', activity_yesterday.max())
print('NEG\n', (activity_yesterday < 0).sum().sort_values(ascending=False))

#Save
activity_yesterday.reset_index(drop=True).to_csv('/UK_BB/brainbody/activity/data/activity_behav_byrecall_i2.csv', index=False)

#### 2.2 Online follow-up: Physical activity yesterday - Instance 3

In [ ]:
#Online follow-up: Physical activity yesterday - Instance 3
activity_behav_i3_i4 = pd.read_csv('/UK_BB/brainbody/activity/data/activity_behav_i3_i4.csv')
activity_yesterday_i3 = activity_behav_i3_i4[['eid',
'Time spent doing vigorous physical activity-3.0',
'Time spent doing moderate physical activity-3.0',
'Time spent doing light physical activity-3.0'
]]

print('Sample size after removing NAs', activity_yesterday_i3.dropna(axis=0).shape)

# Change response encoding
activity_yesterday_i3.loc[:, [
    'Time spent doing vigorous physical activity-3.0',
    'Time spent doing moderate physical activity-3.0']] = activity_yesterday_i3.loc[:, [
    'Time spent doing vigorous physical activity-3.0',
    'Time spent doing moderate physical activity-3.0']].replace({
        #0: None - ok
        10: 1, #Under 10 minutes
        1030: 2, #10-30 minutes
        3060: 3, #30-60 minutes
        12: 4, #1-2 hours
        24: 5, #2-4 hours
        46: 6, #4-6 hours
        600: 7 #6+ hours
})

activity_yesterday_i3.loc[:, [
    'Time spent doing light physical activity-3.0']] = activity_yesterday_i3.loc[:, [
    'Time spent doing light physical activity-3.0']].replace({
        #0: None - ok
        #1: Under 1 hour - ok
        13: 2, #1-3 hours
        35: 3, #3-5 hours
        57: 4, #5-7 hours
        79: 5,  #7-9 hours
        912: 6, #9-12 hours
        1200: 7 #12+ hours
})

activity_yesterday_i3 = activity_yesterday_i3.dropna(axis=0).reset_index(drop=True)

# Remove instance number
activity_yesterday_i3.columns = activity_yesterday_i3.columns.str.replace('-3.0', '')
# Count negative values, min, and max
print('MIN\n', activity_yesterday_i3.min())
print('MAX\n', activity_yesterday_i3.max())
print('NEG\n', (activity_yesterday_i3 < 0).sum().sort_values(ascending=False))
# Save
activity_yesterday_i3.to_csv('/UK_BB/brainbody/activity/data/activity_behav_byrecall_i3.csv', index=False)
activity_yesterday_i3.to_csv('/UK_BB/brainbody/activity/data/activity_byrecall_i3.csv', index=False)

#### 2.3. Online follow-up: Physical activity yesterday - Instance 4 (the latest)

In [ ]:
#Online follow-up: Physical activity yesterday - Instance 4 (the latest)
activity_behav_i3_i4 = pd.read_csv('/UK_BB/brainbody/activity/data/activity_behav_i3_i4.csv')
activity_yesterday_i4 = activity_behav_i3_i4[['eid',
'Time spent doing vigorous physical activity-4.0',
'Time spent doing moderate physical activity-4.0',
'Time spent doing light physical activity-4.0'
]]

print('Sample size after removing NAs', activity_yesterday_i4.dropna(axis=0).shape)

# Change response encoding
activity_yesterday_i4.loc[:, [
    'Time spent doing vigorous physical activity-4.0',
    'Time spent doing moderate physical activity-4.0']] = activity_yesterday_i4.loc[:, [
    'Time spent doing vigorous physical activity-4.0',
    'Time spent doing moderate physical activity-4.0']].replace({
        #0: None - ok
        10: 1, #Under 10 minutes
        1030: 2, #10-30 minutes
        3060: 3, #30-60 minutes
        12: 4, #1-2 hours
        24: 5, #2-4 hours
        46: 6, #4-6 hours
        600: 7 #6+ hours
})

activity_yesterday_i4.loc[:, [
    'Time spent doing light physical activity-4.0']] = activity_yesterday_i4.loc[:, [
    'Time spent doing light physical activity-4.0']].replace({
        #0: None - ok
        #1: Under 1 hour - ok
        13: 2, #1-3 hours
        35: 3, #3-5 hours
        57: 4, #5-7 hours
        79: 5,  #7-9 hours
        912: 6, #9-12 hours
        1200: 7 #12+ hours
})

activity_yesterday_i4 = activity_yesterday_i4.dropna(axis=0).reset_index(drop=True)

# Remove instance number
activity_yesterday_i4.columns = activity_yesterday_i4.columns.str.replace('-4.0', '')
# Count negative values, min, and max
print('MIN\n', activity_yesterday_i4.min())
print('MAX\n', activity_yesterday_i4.max())
print('NEG\n', (activity_yesterday_i4 < 0).sum().sort_values(ascending=False))
# Save
activity_yesterday_i4.to_csv('/UK_BB/brainbody/activity/data/activity_behav_byrecall_i4.csv', index=False)
activity_yesterday_i4.to_csv('/UK_BB/brainbody/activity/data/activity_byrecall_i4.csv', index=False)

# 4. Daily activity (Touchscreen, Lifestyle and environment)

In [ ]:
# Physical activity (Touchscreen)
activity_touchscreen = activity_behav_i2[['eid',
'Usual walking pace-2.0',                 
'Types of transport used (excluding work)-2.0',
'Types of transport used (excluding work)-2.1',
'Types of transport used (excluding work)-2.2',
'Types of transport used (excluding work)-2.3',
'Types of physical activity in last 4 weeks-2.0',
'Types of physical activity in last 4 weeks-2.1',
'Types of physical activity in last 4 weeks-2.2',
'Types of physical activity in last 4 weeks-2.3',
'Types of physical activity in last 4 weeks-2.4',
'Time spent watching television (TV)-2.0',
'Time spent using computer-2.0',
'Time spent driving-2.0',
'Number of days/week walked 10+ minutes-2.0',
'Number of days/week of vigorous physical activity 10+ minutes-2.0',
'Number of days/week of moderate physical activity 10+ minutes-2.0',
'Frequency of walking for pleasure in last 4 weeks-2.0', #NA means 'Walking for pleasure' was not selected as a response to prev. question, NA => 0
'Frequency of strenuous sports in last 4 weeks-2.0', #NA means 'Strenuous sports' was not selected as a response to prev. question, NA => 0 (8771 non-NA)
'Frequency of stair climbing in last 4 weeks-2.0',
'Frequency of other exercises in last 4 weeks-2.0', #NA means 'Other exercises' was not selected as a response to prev. question, NA => 0
'Frequency of light DIY in last 4 weeks-2.0', #NA means 'Light DIY' was not selected as a response to prev. question, NA => 0
'Frequency of heavy DIY in last 4 weeks-2.0', #NA means 'Heavy DIY' was not selected as a response to prev. question, NA => 0
'Duration walking for pleasure-2.0', #NA means 'Walking for pleasure' was not selected as a response to prev. question, NA => 0
'Duration of walks-2.0', #NA means 'Number of days/week walked 10+ minutes' < 1, NA => 0
'Duration of vigorous activity-2.0', #NA means 'Number of days/week of vigorous physical activity 10+ minutes' < 1, NA => 0
'Duration of strenuous sports-2.0', ##NA means 'Strenuous sports' was not selected as a response to prev. question, NA => 0 (8771 non-NA)
'Duration of other exercises-2.0', #NA means 'Other exercises' was not selected as a response to prev. question, NA => 0
'Duration of moderate activity-2.0', #NA means 'Number of days/week of moderate physical activity 10+ minutes' < 1, NA => 0
'Duration of light DIY-2.0', #NA means 'Light DIY' was not selected as a response to prev. question, NA => 0
'Duration of heavy DIY-2.0', #NA means 'Heavy DIY' was not selected as a response to prev. question, NA => 0
'Drive faster than motorway speed limit-2.0'
]]


# Count missing values, negative values, min, and max
print('% missing', ((activity_touchscreen.isna().sum() / len(activity_touchscreen)).round(2)*100).sort_values(ascending=False))
print('MIN\n', activity_touchscreen.min())
print('MAX\n', activity_touchscreen.max())
print('NEG\n', (activity_touchscreen < 0).sum().sort_values(ascending=False))

##### Handle negative values

- -1: Do not know
- -3: Prefer not to answer
____________________________

- -2: Unable to walk ('Number of days/week walked 10+ minutes')
- -10: Less than an hour a day ('Time spent driving', 'Time spent using computer', 'Time spent watching television (TV)')
___________________
- -7: None of the above ('Types of physical activity in last 4 weeks', 'Types of transport used (excluding work)', 'Usual walking pace')

In [ ]:
print('Number of -1 responses\n', (activity_touchscreen == -1).sum().sort_values(ascending=False))
print('Number of -2 responses\n', (activity_touchscreen == -2).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (activity_touchscreen == -3).sum().sort_values(ascending=False))
print('Number of -7 responses\n', (activity_touchscreen == -7).sum().sort_values(ascending=False))
print('Number of -10 responses\n', (activity_touchscreen == -10).sum().sort_values(ascending=False))

In [ ]:
print('Proportion of -1 responses\n', (((activity_touchscreen == -1).sum() / len(activity_touchscreen)).round(4)*100).sort_values(ascending=False))
print('Proportion of -2 responses\n', (((activity_touchscreen == -2).sum() / len(activity_touchscreen)).round(4)*100).sort_values(ascending=False))
print('Proportion of -3 responses\n', (((activity_touchscreen == -3).sum() / len(activity_touchscreen)).round(4)*100).sort_values(ascending=False))
print('Proportion of -7 responses\n', (((activity_touchscreen == -7).sum() / len(activity_touchscreen)).round(4)*100).sort_values(ascending=False))
print('Proportionof -10 responses\n', (((activity_touchscreen == -10).sum() / len(activity_touchscreen)).round(4)*100).sort_values(ascending=False))

Replace -2 ('Unable to walk') with 0 in 'Number of days/week walked 10+ minutes' as it is equivalent to 0 days/weeks walking 10+ minutes


In [ ]:
# Replace -2 ('Unable to walk') with 0
activity_touchscreen.loc[:, :] = activity_touchscreen.loc[:, :].replace({
        -2: 0 #Unable to walk
})
print('Number of -2 responses\n', (activity_touchscreen == -2).sum().sort_values(ascending=False))

Replace -10 (Less than an hour a day) with 0 for:
- Time spent driving
- Time spent using computer
- Time spent watching television (TV)

Units: hours/day

In [ ]:
# Replace -10 ('Less than an hour a day') with 0
activity_touchscreen
activity_touchscreen.loc[:, [
    'Time spent driving-2.0',
    'Time spent using computer-2.0',
    'Time spent watching television (TV)-2.0']] = activity_touchscreen.loc[:, [
    'Time spent driving-2.0',
    'Time spent using computer-2.0',
    'Time spent watching television (TV)-2.0']].replace({
        -10: 0.5, #Under 10 minutes
})

print('Number of -10 responses\n', (activity_touchscreen == -10).sum().sort_values(ascending=False))

In [ ]:
print('Number of -1 responses\n', (activity_touchscreen == -1).sum().sort_values(ascending=False))
print('Number of -2 responses\n', (activity_touchscreen == -2).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (activity_touchscreen == -3).sum().sort_values(ascending=False))
print('Number of -7 responses\n', (activity_touchscreen == -7).sum().sort_values(ascending=False))
print('Number of -10 responses\n', (activity_touchscreen == -10).sum().sort_values(ascending=False))

##### Manage NAs in 'head' questions

In [ ]:
#Drop missing values base on 'head' questions
activity_touchscreen = activity_touchscreen.dropna(subset=[
    'Usual walking pace-2.0',
    'Types of transport used (excluding work)-2.0',
    'Types of physical activity in last 4 weeks-2.0',

    'Number of days/week walked 10+ minutes-2.0',
    'Number of days/week of moderate physical activity 10+ minutes-2.0',
    'Number of days/week of vigorous physical activity 10+ minutes-2.0',

    'Drive faster than motorway speed limit-2.0',
    'Frequency of stair climbing in last 4 weeks-2.0',

    'Time spent driving-2.0',
    'Time spent using computer-2.0',
    'Time spent watching television (TV)-2.0'], axis=0).reset_index(drop=True)

print('Shape after removing missing values from the main questions', activity_touchscreen.shape)
print('NA\n', (activity_touchscreen.isna()).sum().sort_values(ascending=False))

##### Manage NAs in follow-up questions

- Duration of heavy DIY
- Duration of light DIY
- Duration of other exercises
- Duration of strenuous sports
- Duration walking for pleasure
- Frequency of heavy DIY in last 4 weeks
- Frequency of light DIY in last 4 weeks
- Frequency of other exercises in last 4 weeks

were asked if participants indicated that they performed this type of activity for 10 minutes on at least 1 day per week in the previous 4 weeks.

Given that these questions followed the main questions, removing missing values from the head questions ensures and NAs in the follow-up questions mean a negative/none response to the head question.

In [ ]:
activity_touchscreen.to_csv('/UK_BB/brainbody/activity/data/activity_touchscreen_vars_i2.csv', index=False)
#activity_touchscreen = pd.read_csv('/UK_BB/brainbody/activity/data/activity_touchscreen_vars_i2.csv')
activity_touchscreen_encoding = activity_touchscreen.copy()

Types of physical activity in last 4 weeks

In [ ]:
# Types of physical activity in last 4 weeks

#1 Walking for pleasure (not as a means of transport)
#2  Other exercises (eg: swimming, cycling, keep fit, bowling)
#3 Strenuous sports
#4 Light DIY (eg: pruning, watering the lawn)
#5 Heavy DIY (eg: weeding, lawn mowing, carpentry, digging)
#-7 None of the above

walk = [
# Walking for pleasure = 1
'Duration walking for pleasure-2.0',
'Frequency of walking for pleasure in last 4 weeks-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 1, walk] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 1, walk].fillna(0)

other_exercise = [
# Other exercises = 2
'Duration of other exercises-2.0',
'Frequency of other exercises in last 4 weeks-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 2, other_exercise] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 2, other_exercise].fillna(0)

strenuous_sports = [
# Strenuous sports = 3
'Duration of strenuous sports-2.0',
'Frequency of strenuous sports in last 4 weeks-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 3, strenuous_sports] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 3, strenuous_sports].fillna(0)

light_diy = [
# Light DIY = 4
'Duration of light DIY-2.0',
'Frequency of light DIY in last 4 weeks-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 4, light_diy] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 4, light_diy].fillna(0)

###############
heavy_diy = [
# Heavy DIY = 5
'Duration of heavy DIY-2.0',
'Frequency of heavy DIY in last 4 weeks-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 5, heavy_diy] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Types of physical activity in last 4 weeks-2.0'] != 5, heavy_diy].fillna(0)

Number of days/week of moderate physical activity 10+ minutes

In fact, -1 and -3 below would indicate non-respondes who need to be encoded as NAs and followed by NAs in the folowing questions.

In [ ]:
# Number of days/week of moderate physical activity 10+ minutes
moderate_activity = ['Duration of moderate activity-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week of moderate physical activity 10+ minutes-2.0'] == 0, moderate_activity] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week of moderate physical activity 10+ minutes-2.0'] == 0, moderate_activity].fillna(0)

# Number of days/week of vigorous physical activity 10+ minutes
vigorous_activity = ['Duration of vigorous activity-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week of vigorous physical activity 10+ minutes-2.0'] == 0, vigorous_activity] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week of vigorous physical activity 10+ minutes-2.0'] == 0, vigorous_activity].fillna(0)

# Number of days/week walked 10+ minutes
days_walked = ['Duration of walks-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week walked 10+ minutes-2.0'] == 0, days_walked] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week walked 10+ minutes-2.0'] == 0, days_walked].fillna(0)

# Count NA
print('NA\n', (activity_touchscreen_encoding.isna()).sum().sort_values(ascending=False))

'''<0 here would mean that we assign 0, ie no activity, for non-responders, too, but we want to encode them with NA'''

In [ ]:
# Number of days/week of moderate physical activity 10+ minutes
moderate_activity = ['Duration of moderate activity-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week of moderate physical activity 10+ minutes-2.0'] == 0, moderate_activity] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week of moderate physical activity 10+ minutes-2.0'] == 0, moderate_activity].fillna(0)

# Number of days/week of vigorous physical activity 10+ minutes
vigorous_activity = ['Duration of vigorous activity-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week of vigorous physical activity 10+ minutes-2.0'] == 0, vigorous_activity] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week of vigorous physical activity 10+ minutes-2.0'] == 0, vigorous_activity].fillna(0)

# Number of days/week walked 10+ minutes
days_walked = ['Duration of walks-2.0']
activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week walked 10+ minutes-2.0'] == 0, days_walked] = activity_touchscreen_encoding.loc[activity_touchscreen_encoding['Number of days/week walked 10+ minutes-2.0'] == 0, days_walked].fillna(0)

# Count NA
print('NA\n', (activity_touchscreen_encoding.isna()).sum().sort_values(ascending=False))

#### One-hot encode:
- 'Types of physical activity in last 4 weeks'
- 'Types of transport used (excluding work)'
- 'Usual walking pace'

Types of physical activity in last 4 weeks

In [ ]:
# Types of physical activity in last 4 weeks
physical_activity = [
    'Types of physical activity in last 4 weeks-2.0',
    'Types of physical activity in last 4 weeks-2.1',
    'Types of physical activity in last 4 weeks-2.2',
    'Types of physical activity in last 4 weeks-2.3',
    'Types of physical activity in last 4 weeks-2.4']
activity_touchscreen_encoding[physical_activity] = activity_touchscreen_encoding[physical_activity].apply(pd.to_numeric, errors='coerce').astype('Int64')

physical_activity_mapping = {
1: 'Walking for pleasure',
2: 'Other exercises',
3: 'Strenuous sports',
4: 'Light DIY',
5: 'Heavy DIY',
-7: 'Other',
-3: 'Prefer not to answer'
}

dummies = pd.get_dummies(activity_touchscreen_encoding[physical_activity].stack(), prefix='Types of physical activity').groupby(level=0).sum()
dummies.columns = [f'Types of physical activity - {physical_activity_mapping[int(col.split("_")[-1])]}'
                   for col in dummies.columns]

activity_touchscreen_encoding = pd.concat([activity_touchscreen_encoding, dummies], axis=1)
print('Shape', activity_touchscreen_encoding.shape)
# stack() reshapes the DataFrame from wide format to long format. It stacks the columns into a single column, creating a multi-level
# groupby(level=0) function groups the data by the first level of the index (which is the original row index before stacking).
# sum() sums the dummy variables within each group, collapsing the multi-level index back to the original row index.

Types of transport used (excluding work)

In [ ]:
# Types of transport used (excluding work)
transport = [
    'Types of transport used (excluding work)-2.0',
    'Types of transport used (excluding work)-2.1',
    'Types of transport used (excluding work)-2.2',
    'Types of transport used (excluding work)-2.3']
activity_touchscreen_encoding[transport] = activity_touchscreen_encoding[transport].apply(pd.to_numeric, errors='coerce').astype('Int64')

transport_mapping = {
1: 'Car/motor vehicle',
2: 'Walk',
3: 'Public transport',
4: 'Cycle',
-7: 'Other',
-3: 'Prefer not to answer'
}

dummies = pd.get_dummies(activity_touchscreen_encoding[transport].stack(), prefix='Types of transport used').groupby(level=0).sum()
dummies.columns = [f'Types of transport used - {transport_mapping[int(col.split("_")[-1])]}'
                   for col in dummies.columns]

activity_touchscreen_encoding = pd.concat([activity_touchscreen_encoding, dummies], axis=1)
print('Shape', activity_touchscreen_encoding.shape)

In [ ]:
# Usual walking pace
walking_pace = ['Usual walking pace-2.0']
activity_touchscreen_encoding[walking_pace] = activity_touchscreen_encoding[walking_pace].apply(pd.to_numeric, errors='coerce').astype('Int64')

walking_pace_mapping = {
1: 'Slow pace',
2: 'Steady average pace',
3: 'Brisk pace',
-7: 'Other',
-3: 'Prefer not to answer'
}

dummies = pd.get_dummies(activity_touchscreen_encoding[walking_pace].stack(), prefix='Usual walking pace').groupby(level=0).sum()
dummies.columns = [f'Usual walking pace - {walking_pace_mapping[int(col.split("_")[-1])]}'
                   for col in dummies.columns]

activity_touchscreen_encoding = pd.concat([activity_touchscreen_encoding, dummies], axis=1)
print('Shape', activity_touchscreen_encoding.shape)

In [ ]:
# Drop redundant columns
activity_touchscreen_encoding = activity_touchscreen_encoding.drop(columns=[
    'Types of physical activity in last 4 weeks-2.0',
    'Types of physical activity in last 4 weeks-2.1',
    'Types of physical activity in last 4 weeks-2.2',
    'Types of physical activity in last 4 weeks-2.3',
    'Types of physical activity in last 4 weeks-2.4',
    'Types of transport used (excluding work)-2.0',
    'Types of transport used (excluding work)-2.1',
    'Types of transport used (excluding work)-2.2',
    'Types of transport used (excluding work)-2.3',
    'Usual walking pace-2.0']).reset_index(drop=True)

activity_touchscreen_encoding.columns = activity_touchscreen_encoding.columns.str.replace('-2.0', '')
activity_touchscreen_encoding.to_csv('/UK_BB/brainbody/activity/data/activity_touchscreen_i2_with_nonresponders.csv', index=False)

# Count NA and negative values
print('Shape', activity_touchscreen_encoding.shape)
print('NA\n', (activity_touchscreen_encoding.isna()).sum().sort_values(ascending=False))
print('Number of -1 responses\n', (activity_touchscreen_encoding == -1).sum().sort_values(ascending=False))
print('Number of -2 responses\n', (activity_touchscreen_encoding == -2).sum().sort_values(ascending=False))
print('Number of -3 responses\n', (activity_touchscreen_encoding == -3).sum().sort_values(ascending=False))
print('Number of -7 responses\n', (activity_touchscreen_encoding == -7).sum().sort_values(ascending=False))
print('Number of -10 responses\n', (activity_touchscreen_encoding == -10).sum().sort_values(ascending=False))

________________________________________________________________________________________________________
##### Non-responders: 'Do not know' (-1) and 'Prefer not to answer' (-3)

1. [Patterns of item nonresponse behaviour to survey questionnaires are systematic and associated with genetic loci](https://pmc.ncbi.nlm.nih.gov/articles/PMC10444625/)
2. [Dietary assessment in UK Biobank: an evaluation of the performance of the touchscreen dietary questionnaire](https://pmc.ncbi.nlm.nih.gov/articles/PMC5799609/)

'For all questions, participants selecting ‘do not know’, ‘prefer not to answer’ or ‘less than one’ were assigned to separate categories, except for number of bread slices where ‘less than one’ was combined with ‘0’ because of very low numbers for both of these groups.'

3. [Comparison of Sociodemographic and Health-Related Characteristics of UK Biobank Participants With Those of the General Population](https://academic.oup.com/aje/article/186/9/1026/3883629)

'Excludes 2,778 UK Biobank participants aged 40–69 years who were missing data on ethnicity or responded “prefer not to answer” or “do not know.'

4. [Replacement of sedentary behavior with various physical activities and the risk of all-cause and cause-specific mortality](https://pmc.ncbi.nlm.nih.gov/articles/PMC11395964/#Sec2)

'For categorical variables, systematic missing values and responses of “do not know” or “prefer not to answer” were consolidated into the “missing” category.'
________________________________________________________________________________________________________

##### Way 1: Assign NAs to non-responders

In [ ]:
activity_touchscreen = pd.read_csv('/UK_BB/brainbody/activity/data/activity_touchscreen_i2_with_nonresponders.csv')
activity_touchscreen.columns.to_list()

In [ ]:
activity_touchscreen.replace(-1, np.nan, inplace=True)
activity_touchscreen.replace(-3, np.nan, inplace=True)

print('NA\n', (activity_touchscreen.isna()).sum().sort_values(ascending=False))
print('MIN\n', activity_touchscreen.min())
print('NEG\n', (activity_touchscreen < 0).sum().sort_values(ascending=False))

activity_touchscreen.to_csv('/UK_BB/brainbody/activity/data/activity_touchscreen_vars_i2_na.csv', index=False)
activity_touchscreen.to_csv('/UK_BB/brainbody/activity/data/activity_touchscreen.csv', index=False)

##### Final check

In [ ]:
print('Activity yesterday\n')
print('MIN\n', activity_yesterday_i4.min())
print('MAX\n', activity_yesterday_i4.max())
print('NEG\n', (activity_yesterday_i4 < 0).sum().sort_values(ascending=False))
print('NA\n', (activity_yesterday_i4.isna()).sum().sort_values(ascending=False))
print('-----------------------------------------------------------------------')
print('Activity MET\n')
print('MIN\n', activity_met.min())
print('MAX\n', activity_met.max())
print('NEG\n', (activity_met < 0).sum().sort_values(ascending=False))
print('NA\n', (activity_met.isna()).sum().sort_values(ascending=False))
print('-----------------------------------------------------------------------')
print('Activity touchscreen\n')
print('MIN\n', activity_touchscreen.min())
print('MAX\n', activity_touchscreen.max())
print('NEG\n', (activity_touchscreen < 0).sum().sort_values(ascending=False))
print('NA\n', (activity_touchscreen.isna()).sum().sort_values(ascending=False))

##### 2. Drop NAs from non-responders

In [ ]:
activity_touchscreen_nona = activity_touchscreen.dropna(axis=0).reset_index(drop=True)

print('MIN\n', activity_touchscreen_nona.min())
print('NEG\n', (activity_touchscreen_nona < 0).sum().sort_values(ascending=False))
print('NA\n', (activity_touchscreen_nona.isna()).sum().sort_values(ascending=False))

activity_touchscreen_nona.to_csv('/UK_BB/brainbody/activity/data/activity_touchscreen_vars_i2_nona.csv', index=False)
activity_touchscreen_nona.to_csv('/UK_BB/brainbody/activity/data/activity_touchscreen_nona.csv', index=False)